In [ ]:
# ============================================================
# Cell 1: セットアップ（初回のみ実行してください）
# ============================================================
import subprocess
import sys

print("⏳ ライブラリをインストール中...（初回は1〜2分かかります）")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "python-pptx",
    "beautifulsoup4",
    "pillow",
    "lxml",
    "ipywidgets",
    "google-api-python-client",
    "google-auth-httplib2",
    "google-auth-oauthlib",
], check=True)

print("✅ セットアップ完了！次のセルを実行してください。")


⏳ ライブラリをインストール中...（初回は1〜2分かかります）



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✅ セットアップ完了！次のセルを実行してください。


In [ ]:
# ============================================================
# Cell 2: メインUI（何度でも実行できます）
# ============================================================
from google.colab import output
output.enable_custom_widget_manager()
import ipywidgets as widgets
from IPython.display import display, clear_output
from bs4 import BeautifulSoup

# ─────────────────────────────────────────
# ① POTX アップロード
# ─────────────────────────────────────────
potx_uploader = widgets.FileUpload(
    description='📁 POTXをアップロード',
    accept='.potx',
    multiple=False,
    layout=widgets.Layout(width='320px')
)

# ─────────────────────────────────────────
# ② HTML 入力 + 解析ボタン
# ─────────────────────────────────────────
html_input = widgets.Textarea(
    placeholder='ここにHTMLを貼り付けてください...',
    layout=widgets.Layout(width='100%', height='200px')
)

parse_button = widgets.Button(
    description='🔍 HTMLを解析',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

# ─────────────────────────────────────────
# ③ placeholder検出後に動的生成するエリア
# ─────────────────────────────────────────
placeholder_area = widgets.VBox([])

# ─────────────────────────────────────────
# ④ 出力方法選択（最初は非表示）
# ─────────────────────────────────────────
output_format = widgets.RadioButtons(
    options=['PPTXをダウンロード', 'Google Slideとして保存'],
    description='出力方法:',
    layout=widgets.Layout(width='100%', display='none')
)

# ⑤ 変換実行ボタン（最初は非表示）
run_button = widgets.Button(
    description='▶ 変換実行',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px', display='none')
)

# 結果表示エリア
output_area = widgets.Output()

# ─────────────────────────────────────────
# 解析ボタンの処理
# ─────────────────────────────────────────
def on_parse_clicked(b):
    html_str = html_input.value
    if not html_str.strip():
        with output_area:
            clear_output()
            print('⚠️ HTMLを入力してください')
        return

    soup = BeautifulSoup(html_str, 'html.parser')
    placeholders = soup.find_all(attrs={'data-pptx-type': 'placeholder'})

    if placeholders:
        uploaders = []
        for p in placeholders:
            label = p.get('data-placeholder-label', '画像')
            hint = p.get('data-placeholder-hint', '')
            uploader = widgets.FileUpload(
                description=f'🖼️ {label}',
                accept='image/*',
                multiple=False,
                layout=widgets.Layout(width='400px')
            )
            hint_label = widgets.Label(f'　ヒント: {hint}') if hint else widgets.Label('')
            uploader._placeholder_label = label
            uploaders.append(widgets.VBox([uploader, hint_label]))
        placeholder_area.children = [
            widgets.Label('③ 画像をアップロード（placeholderの枠に差し込まれます）'),
            *uploaders
        ]
    else:
        placeholder_area.children = [
            widgets.Label('③ 画像なし（placeholderは検出されませんでした）')
        ]

    output_format.layout.display = ''
    run_button.layout.display = ''

    with output_area:
        clear_output()
        print(f'✅ 解析完了：{len(placeholders)}個のplaceholderを検出しました')

parse_button.on_click(on_parse_clicked)

# ─────────────────────────────────────────
# 変換実行ボタンの処理
# ─────────────────────────────────────────
def on_run_clicked(b):
    with output_area:
        clear_output()

        if not potx_uploader.value:
            print('⚠️ POTXをアップロードしてください')
            return

        html_str = html_input.value
        if not html_str.strip():
            print('⚠️ HTMLを入力してください')
            return

        print('⏳ 変換中...')

        import tempfile, sys
        from pathlib import Path

        # POTXを一時ファイルに保存
        potx_data = list(potx_uploader.value.values())[0]['content']
        tmp_potx = Path(tempfile.mktemp(suffix='.potx'))
        tmp_potx.write_bytes(bytes(potx_data))

        # placeholder画像を一時ファイルに保存
        images = {}
        for child in placeholder_area.children:
            if not isinstance(child, widgets.VBox):
                continue
            uploader_widget = child.children[0]
            if not isinstance(uploader_widget, widgets.FileUpload):
                continue
            if not uploader_widget.value:
                continue
            label = uploader_widget._placeholder_label
            img_data = list(uploader_widget.value.values())[0]['content']
            img_name = list(uploader_widget.value.keys())[0]
            suffix = Path(img_name).suffix or '.jpg'
            tmp_img = Path(tempfile.mktemp(suffix=suffix))
            tmp_img.write_bytes(bytes(img_data))
            images[label] = tmp_img
            print(f'  📎 画像登録: {label} → {tmp_img.name}')

        # converter.pyをimport
        sys.path.insert(0, '/content')
        from converter import convert_html

        # 変換実行
        try:
            pptx_bytes = convert_html(
                html_str=html_str,
                template_path=tmp_potx,
                images=images
            )
        except Exception as e:
            print(f'❌ 変換エラー: {e}')
            return
        finally:
            tmp_potx.unlink(missing_ok=True)
            for p in images.values():
                p.unlink(missing_ok=True)

        print(f'✅ 変換完了！（{len(pptx_bytes):,} bytes）')

        from datetime import datetime, timezone, timedelta
        JST = timezone(timedelta(hours=9))
        filename = f"liber8_ppter_{datetime.now(JST).strftime('%y%m%d%H%M%S')}.pptx"

        if output_format.value == 'PPTXをダウンロード':
            tmp_out = Path(f'/content/{filename}')
            tmp_out.write_bytes(pptx_bytes)
            from google.colab import files
            files.download(str(tmp_out))
            print(f'📥 ダウンロード開始: {filename}')
        else:
            print('⏳ Google Driveに保存中...')
            from google.colab import auth
            auth.authenticate_user()
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseUpload
            import io

            drive_service = build('drive', 'v3')
            file_metadata = {
                'name': filename.replace('.pptx', ''),
                'mimeType': 'application/vnd.google-apps.presentation'
            }
            media = MediaIoBaseUpload(
                io.BytesIO(pptx_bytes),
                mimetype='application/vnd.openxmlformats-officedocument.presentationml.presentation'
            )
            file = drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id,webViewLink'
            ).execute()
            print('✅ Google Slideに保存しました！')
            print(f'🔗 {file.get("webViewLink")}')

run_button.on_click(on_run_clicked)

# ─────────────────────────────────────────
# レイアウト表示
# ─────────────────────────────────────────
display(widgets.VBox([
    widgets.Label('① POTXテンプレートをアップロード'),
    potx_uploader,
    widgets.Label('② HTMLを貼り付けて解析'),
    html_input,
    parse_button,
    placeholder_area,
    output_format,
    run_button,
    output_area,
]))

FileUpload(value=(), accept='.potx', description='📁 POTXをアップロード', layout=Layout(width='300px'))